# broadcasting-rules — worked example 3: Per-cluster weighted centroids without a Python loop

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcasting-rules`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Combining a column-broadcast (insert a size-1 trailing axis so a `(N,)` weight scales `(N, D)` rows) with a broadcast-then-reduce is the workhorse of vectorized aggregation. Inserting the right size-1 axis is what lets a 1-D weight multiply a 2-D feature matrix elementwise before summing.

## Worked solution

Goal: given points `X` of shape `(N, D)` and per-point nonnegative weights `w` of shape `(N,)`, compute the single weighted centroid `sum_i w[i] * X[i] / sum_i w[i]`, shape `(D,)`.

**Step 1 — see why `X * w` fails.** Right-aligning `(N, D)` against `(N,)` tries to match `w`'s only axis against `X`'s trailing `D` axis. Unless `N == D` that errors, and even when it accidentally matches it multiplies the *wrong* axis. We must reshape `w`.

**Step 2 — column-broadcast the weights.** `w[:, None]` (equivalently `w.unsqueeze(1)`) gives shape `(N, 1)`. Now `X * w[:, None]` right-aligns `(N, D)` against `(N, 1)`: the `1` stretches across the `D` feature axis, so every feature of row `i` is scaled by `w[i]`. Result shape `(N, D)`.

**Step 3 — reduce over points.** `.sum(dim=0)` collapses the `N` axis, giving the weighted feature sum of shape `(D,)`.

**Step 4 — normalize.** Divide by `w.sum()`, a scalar, which broadcasts trivially. The result is the weighted mean position, shape `(D,)`.

We cross-check against an explicit Python loop to prove the broadcast did the same arithmetic.

In [ ]:
def weighted_centroid(X, w):
    weighted = X * w[:, None]      # (N, D), each row scaled by w[i]
    return weighted.sum(dim=0) / w.sum()

t.manual_seed(0)
X = t.randn(6, 4)
w = t.rand(6)
c = weighted_centroid(X, w)
# Independent loop reference.
acc = t.zeros(4)
for i in range(X.shape[0]):
    acc += w[i] * X[i]
ref = acc / w.sum()
print('centroid shape =', tuple(c.shape))
print('matches loop   =', bool(t.allclose(c, ref, atol=1e-6)))